In [278]:
import torch
from time import time

In [279]:
def testCode(function, tensor: torch.Tensor, nTime: int):
    start = time()
    function(tensor, nTime)
    time_of_calc = time() - start
    return time_of_calc

In [280]:
def oneBoundary(tensor: torch.Tensor, nTime: int):
    result = []
    for _ in range(nTime):
        temp = tensor
        temp = torch.abs(temp)
        
        max = torch.max(torch.max(temp, dim=-1).values)
        result.append(max)
    return result

In [284]:
def boundariesVanilla(tensor: torch.Tensor, nTime: int):
    result = []
    for _ in range(nTime):
        temp = tensor
        temp = torch.abs(temp)
        
        max = torch.max(torch.max(temp, dim=-1).values)
        temp = torch.where(temp == 0, 10000, temp)
        min = torch.min(torch.min(temp, dim=-1).values)
        result.append([min, max])
    return result

In [285]:
def boundariesModified(tensor: torch.Tensor, nTime: int):
    eps = torch.tensor(torch.finfo(torch.bfloat16).tiny, dtype=torch.bfloat16).cuda()
    result = []
    for _ in range(nTime):
        temp = tensor
        temp = torch.abs(temp)

        y = temp - eps
        out = torch.max(torch.max(torch.stack((y, 1 / y), dim=1), dim=0).values, dim=-1).values
        stacked = torch.stack((y, 1 / y), dim=1)
        out = torch.max(torch.max(stacked, dim=0).values, dim=-1).values
        min = 1 / out[1]
        max = out[0]
        result.append([min, max])
    return result

In [293]:
def boundariesModified_v2(tensor: torch.Tensor, nTime: int):
    eps = torch.tensor(torch.finfo(torch.bfloat16).tiny, dtype=torch.bfloat16).cuda()
    result = []
    for _ in range(nTime):
        temp = tensor
        temp = torch.abs(temp)

        min, max = torch.aminmax(temp.add_(torch.exp(-pow(temp,2)/eps) * temp[(0,)*temp.dim()]))
        # min, max = torch.aminmax(temp.add_(temp.pow_(2).div_(-eps).exp_() * temp[(0,)*temp.dim()]))
        result.append([min, max])
    return result

# temp.add_(temp.pow_(2).div_(-eps).exp_() * temp[(0,)*temp.dim()])

In [294]:
N, M = 1000, 20000
nTime = 10

input = torch.round(torch.normal(mean = 0, std=10, size=(N, M), dtype=torch.bfloat16), decimals=2).cuda()
oneBoundary_ = testCode(oneBoundary, input, nTime)
print("Time of calculation for one boundary: " + str(oneBoundary_) + " s")

input = torch.round(torch.normal(mean = 0, std=10, size=(N, M), dtype=torch.bfloat16), decimals=2).cuda()
boundariesVanilla_ = testCode(boundariesVanilla, input, nTime)
print("Time of calculation for boundariesVanilla: " + str(boundariesVanilla_) + " s")
print("overhead for vanilla: " + str(((boundariesVanilla_ / oneBoundary_) - 1) * 100) + " %")

input = torch.round(torch.normal(mean = 0, std=10, size=(N, M), dtype=torch.bfloat16), decimals=2).cuda()
boundariesModified_ = testCode(boundariesModified, input, nTime)
print("Time of calculation for boundariesModified: " + str(boundariesModified_) + " s")
print("overhead for boundariesModified: " + str(((boundariesModified_ / oneBoundary_) - 1) * 100) + " %")

input = torch.round(torch.normal(mean = 0, std=10, size=(N, M), dtype=torch.bfloat16), decimals=2).cuda()
boundariesModified_v2_ = testCode(boundariesModified_v2, input, nTime)
print("Time of calculation for boundariesModified_v2: " + str(boundariesModified_v2_) + " s")
print("overhead for boundariesModified_v2: " + str(((boundariesModified_v2_ / oneBoundary_) - 1) * 100) + " %")

Time of calculation for one boundary: 0.0010123252868652344 s
Time of calculation for boundariesVanilla: 0.002561807632446289 s
overhead for vanilla: 153.061705134244 %
Time of calculation for boundariesModified: 0.004789590835571289 s
overhead for boundariesModified: 373.12764955252 %
Time of calculation for boundariesModified_v2: 0.003484964370727539 s
overhead for boundariesModified_v2: 244.2534149788036 %


In [268]:
torch.finfo(torch.bfloat16).tiny
x = torch.rand(5,5).bfloat16() / 1000


min = torch.min(x)
x[torch.where(x == torch.max(x))] = 0
max = torch.max(x)
# ----------------------------------------------------------------------------
print("result for vanilla:")
print("min: " + str(min))
print("max: " + str(max))

# ----------------------------------------------------------------------------
eps = torch.tensor(torch.finfo(torch.bfloat16).tiny, dtype=torch.bfloat16)
y = x - eps
min_ = 1 / torch.max(1 / y)
max_ = torch.max(y)

print("\nresult for modification (two calling max):")
print("min: " + str(min_))
print("max: " + str(max_))

# ----------------------------------------------------------------------------
# out = torch.max(torch.max(torch.stack((y, 1 / y), dim=1), dim=0).values, dim=-1).values
# min_mod = 1 / out[1]
# max_mod = out[0]

# print(x)
# print(torch.exp(-y * y/eps) * y[(0,)*y.dim()])
min_mod, max_mod = torch.aminmax(y.add_(torch.exp(-pow(y,2)/eps) * y[(0,)*y.dim()]))
# min_mod, max_mod = torch.aminmax(torch.exp(y))
# print(min_mod)
# print(max_mod)

 
# print("\nresult for modification (one calling max):")
# print("min: " + str(min_mod))
# print("max: " + str(max_mod))

result for vanilla:
min: tensor(3.3617e-05, dtype=torch.bfloat16)
max: tensor(0.0010, dtype=torch.bfloat16)

result for modification (two calling max):
min: tensor(3.3617e-05, dtype=torch.bfloat16)
max: tensor(0.0010, dtype=torch.bfloat16)


In [222]:
y = torch.tensor([0.5, 2.0, 0.25, 4.0])
max_mod = torch.max(y, 1 / y)
min_mod = torch.min(y, 1 / y)